In [ ]:
import pandas as pd
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'README.md').exists() and (p / 'data').exists():
            return p
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(EXTERNAL_DIR / 'SD_CANTAB_DB_public.csv')
print(df.shape)
df.head()

In [ ]:
df = pd.read_csv(EXTERNAL_DIR / 'SD_CANTAB_DB_public.csv')
print(df.shape)
df.head()

In [ ]:
df = pd.read_csv(EXTERNAL_DIR / 'SD_CANTAB_DB_public.csv')

In [ ]:
import pandas as pd

df = pd.read_csv(EXTERNAL_DIR / 'SD_CANTAB_DB_public.csv')
print(df.shape)
df.head()

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(EXTERNAL_DIR / 'SD_CANTAB_DB_public.csv')

drop_cols = ['Unnamed: 108', 'Unnamed: 110', 'Notes']
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

df['condition'] = df['Subject ID'].str.extract(r'_(before|after)$')[0]
df['sleep_deprived'] = (df['condition'] == 'after').astype(int)

df['subject_base'] = df['Subject ID'].str.replace(r'_(before|after)$', '', regex=True)

print(df[['Subject ID', 'subject_base', 'condition', 'sleep_deprived']].head())
print(df['condition'].value_counts(dropna=False))
print(df.shape)

In [ ]:
status_cols = [c for c in df.columns if 'Status' in c]
non_feature_cols = ['Subject ID', 'condition', 'sleep_deprived', 'subject_base']

feature_df = df.drop(columns=status_cols + [c for c in non_feature_cols if c in df.columns], errors='ignore')

feature_df = feature_df.apply(pd.to_numeric, errors='coerce')

print(feature_df.shape)
feature_df.head()

In [ ]:
missing = feature_df.isna().mean().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X = feature_df.copy()
y = df['sleep_deprived'].copy()

logreg = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000))
])

rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=1,
        random_state=42
    ))
])

loo = LeaveOneOut()

logreg_scores = cross_val_score(logreg, X, y, cv=loo, scoring='accuracy')
rf_scores = cross_val_score(rf, X, y, cv=loo, scoring='accuracy')

print('LogReg LOO accuracy:', logreg_scores.mean())
print('RF LOO accuracy:', rf_scores.mean())

In [ ]:
summary = df.groupby('condition')[feature_df.columns].mean().T
summary['diff_after_minus_before'] = summary['after'] - summary['before']
summary['abs_diff'] = summary['diff_after_minus_before'].abs()

summary = summary.sort_values('abs_diff', ascending=False)
summary.head(20)

In [ ]:
import matplotlib.pyplot as plt

top_diff = summary.head(10).sort_values('diff_after_minus_before')

plt.figure(figsize=(8, 5))
plt.barh(top_diff.index, top_diff['diff_after_minus_before'])
plt.xlabel('After - Before')
plt.title('Top Cognitive Metrics Changed After Sleep Deprivation')
plt.tight_layout()
plt.show()

In [ ]:
paired = df[['subject_base', 'condition'] + feature_df.columns.tolist()].copy()

before_df = paired[paired['condition'] == 'before'].drop(columns=['condition'])
after_df = paired[paired['condition'] == 'after'].drop(columns=['condition'])

paired_delta = before_df.merge(after_df, on='subject_base', suffixes=('_before', '_after'))

print(paired_delta.shape)
paired_delta.head()

## Preliminary interpretation

This dataset contains cognitive test measurements collected before and after 24 hours of sleep deprivation. As a first step, the analysis tests whether CANTAB-derived cognitive metrics can distinguish baseline from sleep-deprived sessions.

In addition to classification, mean differences across conditions are examined to identify which cognitive measures appear most affected by sleep deprivation.

In [ ]:
import pandas as pd
import numpy as np

paired = df[['subject_base', 'condition'] + feature_df.columns.tolist()].copy()

before_df = paired[paired['condition'] == 'before'].drop(columns=['condition'])
after_df = paired[paired['condition'] == 'after'].drop(columns=['condition'])

paired_df = before_df.merge(after_df, on='subject_base', suffixes=('_before', '_after'))

print("Paired subjects:", paired_df.shape[0])
paired_df.head()

In [ ]:
from scipy.stats import ttest_rel, wilcoxon, shapiro

def paired_test(before, after):
    tmp = pd.DataFrame({'before': before, 'after': after}).dropna()

    n = len(tmp)
    if n < 3:
        return {
            'n': n,
            'test_used': None,
            'statistic': np.nan,
            'p_value': np.nan,
            'mean_before': tmp['before'].mean() if n > 0 else np.nan,
            'mean_after': tmp['after'].mean() if n > 0 else np.nan,
            'mean_diff': (tmp['after'] - tmp['before']).mean() if n > 0 else np.nan,
            'std_diff': (tmp['after'] - tmp['before']).std(ddof=1) if n > 1 else np.nan,
            'effect_size_dz': np.nan,
            'normality_p': np.nan
        }

    diff = tmp['after'] - tmp['before']

    
    if n >= 3:
        normality_p = shapiro(diff).pvalue
    else:
        normality_p = np.nan

    mean_before = tmp['before'].mean()
    mean_after = tmp['after'].mean()
    mean_diff = diff.mean()
    std_diff = diff.std(ddof=1)

    
    effect_size_dz = mean_diff / std_diff if std_diff and not np.isclose(std_diff, 0) else np.nan

    
    if pd.notna(normality_p) and normality_p >= 0.05:
        stat, p = ttest_rel(tmp['after'], tmp['before'], nan_policy='omit')
        test_used = 'paired_t_test'
    else:
        try:
            stat, p = wilcoxon(tmp['after'], tmp['before'])
            test_used = 'wilcoxon'
        except ValueError:
            stat, p = np.nan, np.nan
            test_used = 'wilcoxon_failed'

    return {
        'n': n,
        'test_used': test_used,
        'statistic': stat,
        'p_value': p,
        'mean_before': mean_before,
        'mean_after': mean_after,
        'mean_diff': mean_diff,
        'std_diff': std_diff,
        'effect_size_dz': effect_size_dz,
        'normality_p': normality_p
    }

In [ ]:
results = []

for col in feature_df.columns:
    before_col = f'{col}_before'
    after_col = f'{col}_after'

    if before_col in paired_df.columns and after_col in paired_df.columns:
        res = paired_test(paired_df[before_col], paired_df[after_col])
        res['metric'] = col
        results.append(res)

paired_results_df = pd.DataFrame(results)

paired_results_df = paired_results_df.sort_values(['p_value', 'mean_diff'], ascending=[True, False])
paired_results_df.head(20)

In [ ]:
from statsmodels.stats.multitest import multipletests

valid_mask = paired_results_df['p_value'].notna()

rej, p_adj, _, _ = multipletests(
    paired_results_df.loc[valid_mask, 'p_value'],
    method='fdr_bh'
)

paired_results_df.loc[valid_mask, 'p_value_fdr_bh'] = p_adj
paired_results_df.loc[valid_mask, 'significant_fdr_0_05'] = rej

paired_results_df = paired_results_df.sort_values('p_value_fdr_bh', ascending=True)
paired_results_df.head(20)

In [ ]:
%pip install statsmodels

In [ ]:
from statsmodels.stats.multitest import multipletests

In [ ]:
from statsmodels.stats.multitest import multipletests

valid_mask = paired_results_df['p_value'].notna()

rej, p_adj, _, _ = multipletests(
    paired_results_df.loc[valid_mask, 'p_value'],
    method='fdr_bh'
)

paired_results_df.loc[valid_mask, 'p_value_fdr_bh'] = p_adj
paired_results_df.loc[valid_mask, 'significant_fdr_0_05'] = rej

paired_results_df = paired_results_df.sort_values('p_value_fdr_bh', ascending=True)
paired_results_df.head(20)

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import ttest_rel, wilcoxon, shapiro
from statsmodels.stats.multitest import multipletests


if 'df' not in globals():
    raise NameError("df is not defined. Load and clean the SD_CANTAB_DB_public.csv file first.")

if 'feature_df' not in globals():
    raise NameError("feature_df is not defined. Run the feature selection/cleaning step first.")


paired = df[['subject_base', 'condition'] + feature_df.columns.tolist()].copy()

before_df = paired[paired['condition'] == 'before'].drop(columns=['condition'])
after_df = paired[paired['condition'] == 'after'].drop(columns=['condition'])

paired_df = before_df.merge(after_df, on='subject_base', suffixes=('_before', '_after'))

print("Paired subjects:", paired_df.shape[0])


def paired_test(before, after):
    tmp = pd.DataFrame({'before': before, 'after': after}).dropna()

    n = len(tmp)
    if n < 3:
        return {
            'n': n,
            'test_used': None,
            'statistic': np.nan,
            'p_value': np.nan,
            'mean_before': tmp['before'].mean() if n > 0 else np.nan,
            'mean_after': tmp['after'].mean() if n > 0 else np.nan,
            'mean_diff': (tmp['after'] - tmp['before']).mean() if n > 0 else np.nan,
            'std_diff': (tmp['after'] - tmp['before']).std(ddof=1) if n > 1 else np.nan,
            'effect_size_dz': np.nan,
            'normality_p': np.nan
        }

    diff = tmp['after'] - tmp['before']
    mean_before = tmp['before'].mean()
    mean_after = tmp['after'].mean()
    mean_diff = diff.mean()
    std_diff = diff.std(ddof=1)

    effect_size_dz = mean_diff / std_diff if pd.notna(std_diff) and not np.isclose(std_diff, 0) else np.nan

    if np.isclose(diff.std(ddof=1), 0):
        normality_p = np.nan

        if np.isclose(mean_diff, 0):
            return {
                'n': n,
                'test_used': 'constant_no_change',
                'statistic': 0.0,
                'p_value': 1.0,
                'mean_before': mean_before,
                'mean_after': mean_after,
                'mean_diff': mean_diff,
                'std_diff': std_diff,
                'effect_size_dz': 0.0,
                'normality_p': normality_p
            }
        else:
            return {
                'n': n,
                'test_used': 'constant_change',
                'statistic': np.nan,
                'p_value': np.nan,
                'mean_before': mean_before,
                'mean_after': mean_after,
                'mean_diff': mean_diff,
                'std_diff': std_diff,
                'effect_size_dz': np.nan,
                'normality_p': normality_p
            }

    normality_p = shapiro(diff).pvalue

    if normality_p >= 0.05:
        stat, p = ttest_rel(tmp['after'], tmp['before'], nan_policy='omit')
        test_used = 'paired_t_test'
    else:
        try:
            stat, p = wilcoxon(tmp['after'], tmp['before'])
            test_used = 'wilcoxon'
        except ValueError:
            stat, p = np.nan, np.nan
            test_used = 'wilcoxon_failed'

    return {
        'n': n,
        'test_used': test_used,
        'statistic': stat,
        'p_value': p,
        'mean_before': mean_before,
        'mean_after': mean_after,
        'mean_diff': mean_diff,
        'std_diff': std_diff,
        'effect_size_dz': effect_size_dz,
        'normality_p': normality_p
    }

results = []

for col in feature_df.columns:
    before_col = f'{col}_before'
    after_col = f'{col}_after'

    if before_col in paired_df.columns and after_col in paired_df.columns:
        res = paired_test(paired_df[before_col], paired_df[after_col])
        res['metric'] = col
        results.append(res)

paired_results_df = pd.DataFrame(results)

valid_mask = paired_results_df['p_value'].notna()

if valid_mask.sum() > 0:
    rej, p_adj, _, _ = multipletests(
        paired_results_df.loc[valid_mask, 'p_value'],
        method='fdr_bh'
    )
    paired_results_df.loc[valid_mask, 'p_value_fdr_bh'] = p_adj
    paired_results_df.loc[valid_mask, 'significant_fdr_0_05'] = rej
else:
    paired_results_df['p_value_fdr_bh'] = np.nan
    paired_results_df['significant_fdr_0_05'] = np.nan

paired_results_df = paired_results_df.sort_values(
    by=['p_value_fdr_bh', 'p_value'],
    ascending=[True, True],
    na_position='last'
)

summary_cols = [
    'metric',
    'n',
    'test_used',
    'mean_before',
    'mean_after',
    'mean_diff',
    'effect_size_dz',
    'p_value',
    'p_value_fdr_bh',
    'significant_fdr_0_05'
]

paired_summary = paired_results_df[summary_cols].copy()

print("paired_df shape:", paired_df.shape)
print("paired_results_df shape:", paired_results_df.shape)

display(paired_summary.head(20))

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(EXTERNAL_DIR / 'SD_CANTAB_DB_public.csv')

drop_cols = ['Unnamed: 108', 'Unnamed: 110', 'Notes']
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

df['condition'] = df['Subject ID'].str.extract(r'_(before|after)$')[0]
df['sleep_deprived'] = (df['condition'] == 'after').astype(int)
df['subject_base'] = df['Subject ID'].str.replace(r'_(before|after)$', '', regex=True)

print(df.shape)
df[['Subject ID', 'subject_base', 'condition', 'sleep_deprived']].head()

In [ ]:
status_cols = [c for c in df.columns if 'Status' in c]
non_feature_cols = ['Subject ID', 'condition', 'sleep_deprived', 'subject_base']

feature_df = df.drop(columns=status_cols + [c for c in non_feature_cols if c in df.columns], errors='ignore')
feature_df = feature_df.apply(pd.to_numeric, errors='coerce')

print(feature_df.shape)
feature_df.head()

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import ttest_rel, wilcoxon, shapiro
from statsmodels.stats.multitest import multipletests


if 'df' not in globals():
    raise NameError("df is not defined. Load and clean the SD_CANTAB_DB_public.csv file first.")

if 'feature_df' not in globals():
    raise NameError("feature_df is not defined. Run the feature selection/cleaning step first.")


paired = df[['subject_base', 'condition'] + feature_df.columns.tolist()].copy()

before_df = paired[paired['condition'] == 'before'].drop(columns=['condition'])
after_df = paired[paired['condition'] == 'after'].drop(columns=['condition'])

paired_df = before_df.merge(after_df, on='subject_base', suffixes=('_before', '_after'))

print("Paired subjects:", paired_df.shape[0])

def paired_test(before, after):
    tmp = pd.DataFrame({'before': before, 'after': after}).dropna()

    n = len(tmp)
    if n < 3:
        return {
            'n': n,
            'test_used': None,
            'statistic': np.nan,
            'p_value': np.nan,
            'mean_before': tmp['before'].mean() if n > 0 else np.nan,
            'mean_after': tmp['after'].mean() if n > 0 else np.nan,
            'mean_diff': (tmp['after'] - tmp['before']).mean() if n > 0 else np.nan,
            'std_diff': (tmp['after'] - tmp['before']).std(ddof=1) if n > 1 else np.nan,
            'effect_size_dz': np.nan,
            'normality_p': np.nan
        }

    diff = tmp['after'] - tmp['before']
    mean_before = tmp['before'].mean()
    mean_after = tmp['after'].mean()
    mean_diff = diff.mean()
    std_diff = diff.std(ddof=1)

    effect_size_dz = mean_diff / std_diff if pd.notna(std_diff) and not np.isclose(std_diff, 0) else np.nan

    if np.isclose(diff.std(ddof=1), 0):
        normality_p = np.nan

        if np.isclose(mean_diff, 0):
            return {
                'n': n,
                'test_used': 'constant_no_change',
                'statistic': 0.0,
                'p_value': 1.0,
                'mean_before': mean_before,
                'mean_after': mean_after,
                'mean_diff': mean_diff,
                'std_diff': std_diff,
                'effect_size_dz': 0.0,
                'normality_p': normality_p
            }
        else:
            return {
                'n': n,
                'test_used': 'constant_change',
                'statistic': np.nan,
                'p_value': np.nan,
                'mean_before': mean_before,
                'mean_after': mean_after,
                'mean_diff': mean_diff,
                'std_diff': std_diff,
                'effect_size_dz': np.nan,
                'normality_p': normality_p
            }

    normality_p = shapiro(diff).pvalue

    if normality_p >= 0.05:
        stat, p = ttest_rel(tmp['after'], tmp['before'], nan_policy='omit')
        test_used = 'paired_t_test'
    else:
        try:
            stat, p = wilcoxon(tmp['after'], tmp['before'])
            test_used = 'wilcoxon'
        except ValueError:
            stat, p = np.nan, np.nan
            test_used = 'wilcoxon_failed'

    return {
        'n': n,
        'test_used': test_used,
        'statistic': stat,
        'p_value': p,
        'mean_before': mean_before,
        'mean_after': mean_after,
        'mean_diff': mean_diff,
        'std_diff': std_diff,
        'effect_size_dz': effect_size_dz,
        'normality_p': normality_p
    }


results = []

for col in feature_df.columns:
    before_col = f'{col}_before'
    after_col = f'{col}_after'

    if before_col in paired_df.columns and after_col in paired_df.columns:
        res = paired_test(paired_df[before_col], paired_df[after_col])
        res['metric'] = col
        results.append(res)

paired_results_df = pd.DataFrame(results)

valid_mask = paired_results_df['p_value'].notna()

if valid_mask.sum() > 0:
    rej, p_adj, _, _ = multipletests(
        paired_results_df.loc[valid_mask, 'p_value'],
        method='fdr_bh'
    )
    paired_results_df.loc[valid_mask, 'p_value_fdr_bh'] = p_adj
    paired_results_df.loc[valid_mask, 'significant_fdr_0_05'] = rej
else:
    paired_results_df['p_value_fdr_bh'] = np.nan
    paired_results_df['significant_fdr_0_05'] = np.nan

paired_results_df = paired_results_df.sort_values(
    by=['p_value_fdr_bh', 'p_value'],
    ascending=[True, True],
    na_position='last'
)

summary_cols = [
    'metric',
    'n',
    'test_used',
    'mean_before',
    'mean_after',
    'mean_diff',
    'effect_size_dz',
    'p_value',
    'p_value_fdr_bh',
    'significant_fdr_0_05'
]

paired_summary = paired_results_df[summary_cols].copy()

print("paired_df shape:", paired_df.shape)
print("paired_results_df shape:", paired_results_df.shape)

display(paired_summary.head(20))

In [ ]:
paired_results_df.sort_values('effect_size_dz', key=lambda s: s.abs(), ascending=False).head(20)

In [ ]:
import matplotlib.pyplot as plt

top_metrics = (
    paired_results_df
    .sort_values('p_value', ascending=True)
    .head(6)['metric']
    .tolist()
)

fig, axes = plt.subplots(len(top_metrics), 1, figsize=(8, 3 * len(top_metrics)))

if len(top_metrics) == 1:
    axes = [axes]

for ax, metric in zip(axes, top_metrics):
    before_col = f'{metric}_before'
    after_col = f'{metric}_after'

    tmp = paired_df[['subject_base', before_col, after_col]].dropna()

    for _, row in tmp.iterrows():
        ax.plot(['Before', 'After'], [row[before_col], row[after_col]], marker='o', alpha=0.7)

    ax.set_title(metric)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
top10 = paired_results_df.nsmallest(10, 'p_value').copy()
top10 = top10.sort_values('mean_diff')

plt.figure(figsize=(9, 6))
plt.barh(top10['metric'], top10['mean_diff'])
plt.xlabel('Mean change (After - Before)')
plt.title('Top Paired Changes After Sleep Deprivation')
plt.tight_layout()
plt.show()

In [ ]:
summary_cols = [
    'metric',
    'n',
    'test_used',
    'mean_before',
    'mean_after',
    'mean_diff',
    'effect_size_dz',
    'p_value',
    'p_value_fdr_bh',
    'significant_fdr_0_05'
]

paired_summary = paired_results_df[summary_cols].copy()
paired_summary.head(20)

## Paired analysis interpretation

The paired before-vs-after analysis revealed the strongest exploratory changes in RVP- and RTI-related cognitive metrics after 24 hours of sleep deprivation.

Although no metric remained significant after FDR correction, several raw p-values and within-subject effect sizes suggested consistent changes in sustained attention and reaction-time performance. In particular, RVP-related accuracy/hit measures decreased after sleep deprivation, while several RTI timing-related measures increased, indicating slower cognitive-motor responses.

Given the small sample size, these findings should be interpreted as exploratory but directionally informative.

## Conclusion

This case study suggests that 24-hour sleep deprivation leaves a measurable signature in cognitive performance, especially in sustained attention and reaction-time-related metrics.

The strongest exploratory changes were observed in RVP and RTI measures. However, after correction for multiple comparisons, no effect remained statistically significant, which is consistent with the small sample size of the dataset.

Overall, the analysis supports an exploratory interpretation: cognitive performance appears sensitive to acute sleep deprivation, but larger datasets would be needed for robust confirmatory inference.

In [ ]:
paired_summary.to_csv(PROCESSED_DIR / 'paired_summary.csv', index=False)
paired_results_df.to_csv(PROCESSED_DIR / 'paired_results_df.csv', index=False)

print('Saved: paired_summary.csv')
print('Saved: paired_results_df.csv')

In [ ]:
paired_summary.to_csv(PROCESSED_DIR / 'paired_summary.csv', index=False)
paired_results_df.to_csv(PROCESSED_DIR / 'paired_results_df.csv', index=False)

print('Saved:', PROCESSED_DIR / 'paired_summary.csv')
print('Saved:', PROCESSED_DIR / 'paired_results_df.csv')